In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import sqlite3
import pandas as pd
import re
import time
import os

D_DRIVE_CACHE = "/mnt/d/huggingface_cache"
model_id = "Snowflake/Arctic-Text2SQL-R1-7B"

def clean_generated_sql(raw_text):
    clean = re.sub(r"```sql|```", "", raw_text)
    select_match = re.search(r"\bSELECT\b", clean, re.IGNORECASE)
    if select_match:
        clean = clean[select_match.start():]
    return clean.split(';')[0].split('\n\n')[0].strip()

print(f"Loading model into {D_DRIVE_CACHE}...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    cache_dir=D_DRIVE_CACHE
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    cache_dir=D_DRIVE_CACHE
)

with open("tables.json", "r") as f:
    tables = json.load(f)

db_map = {entry['db_id']: ", ".join([f"{entry['table_names_original'][c[0]]}.{c[1]}" 
          for c in entry['column_names_original'] if c[0] != -1]) for entry in tables}

with open("dev.json", "r") as f:
    dev_data = json.load(f)

results_log = []
num_to_test = 1000 

for i in range(num_to_test):
    item = dev_data[i]
    db_id, question, gold_sql = item['db_id'], item['question'], item['query']
    db_path = f"database/{db_id}/{db_id}.sqlite"
    schema = db_map.get(db_id, "")
    
    prompt = f"### Instruction:\nGenerate SQL for: {question}\nSchema: {schema}\n\n### Response:\nSELECT"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    latency = time.time() - start_time
    
    tps = (len(outputs[0]) - len(inputs.input_ids[0])) / latency
    raw_gen = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    pred_sql = clean_generated_sql("SELECT " + raw_gen)

    report = {"question": question, "gold": gold_sql, "pred": pred_sql, "match": False, "tps": tps}
    try:
        conn = sqlite3.connect(db_path)
        p_df = pd.read_sql_query(pred_sql, conn)
        g_df = pd.read_sql_query(gold_sql, conn)
        report["match"] = p_df.equals(g_df)
        conn.close()
    except: 
        pass
        
    results_log.append(report)
    print(f"[{i+1}/{num_to_test}] Match: {report['match']} | Speed: {tps:.2f} t/s")

print(f"\nAverage Speed: {sum(r['tps'] for r in results_log)/len(results_log):.2f} tokens/sec")